run this headless

conda activate guitarmidi
screen  jupyter nbconvert --to notebook --execute traning.ipynb --output=training_out.ipynb --ExecutePreprocessor.timeout=-1 > nbconvert.log 2>&1 &


In [ ]:
import tensorflow as tf
from tensorflow.keras import layers, models, optimizers, Model
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint, Callback,ReduceLROnPlateau
from model import build_cnn_model
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import clear_output
import os
import glob # To list files
import time
from IPython.display import clear_output
from datetime import datetime # Import datetime
from common import INPUT_SHAPE,OUTPUT_DIM_NOTES,OUTPUT_DIM_ONSETS
# --- Essential for GPU memory management ---
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        tf.keras.mixed_precision.set_global_policy('mixed_float16')
        print("Mixed precision policy set to 'mixed_float16'.")

        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True) # Corrected
        print("Memory growth enabled for GPUs.")
    except RuntimeError as e:
        print(f"Error configuring GPU: {e}")
# ---------------------------------------------------------------------------------

print(f"TensorFlow version: {tf.__version__}")

# --- Configuration (using values from serialization part) ---

LEARNING_RATE = 0.001
BATCH_SIZE = 16 # Adjust as needed
EPOCHS = 100

# Directories where slices were saved
input_data_dir = 'data_slices/input'
output_data_dir = 'data_slices/output' # This will be for the 'note' labels
onsets_data_dir = 'data_slices/onsets' # This will be for the 'onsets' labels

# --- Custom Callback for Live Loss Plotting (remains the same, but will show two sets of metrics) ---
class JupyterLivePlottingCallback(Callback):
    def __init__(self, fig_title="Training Metrics", base_plot_dir="training_plots"):
        super().__init__()
        self.fig_title = fig_title
        
        # Generate a timestamp for the directory name
        timestamp = datetime.now().strftime("%Y%m%d-%H%M%S")
        self.plot_save_dir = os.path.join(base_plot_dir, f"run_{timestamp}")
        
        # Create the directory if it doesn't exist
        os.makedirs(self.plot_save_dir, exist_ok=True)
        print(f"Plots will be saved to: {self.plot_save_dir}")

        self.epoch_data = {
            'loss': [], 'note_output_loss': [], 'onsets_output_loss': [],
            'note_output_accuracy': [], 'onsets_output_accuracy': [],
            'val_loss': [], 'val_note_output_loss': [], 'val_onsets_output_loss': [],
            'val_note_output_accuracy': [], 'val_onsets_output_accuracy': []
        }
        self.epochs = []

    def on_train_begin(self, logs=None):
        self.epoch_data = {k: [] for k in self.epoch_data.keys()}
        self.epochs = []
        print("Starting Keras model training with live plot. Output will update below...")

    def on_epoch_end(self, epoch, logs=None):
        clear_output(wait=True)

        self.epochs.append(epoch + 1)
        for key in self.epoch_data:
            self.epoch_data[key].append(logs.get(key))

        fig, axes = plt.subplots(2, 2, figsize=(16, 12))
        fig.suptitle(self.fig_title)

        # Plot Note Accuracy
        axes[0, 0].plot(self.epochs, self.epoch_data['note_output_accuracy'], 'b-o', label='Training Note Accuracy')
        axes[0, 0].plot(self.epochs, self.epoch_data['val_note_output_accuracy'], 'r-x', label='Validation Note Accuracy')
        axes[0, 0].set_title('Note Accuracy')
        axes[0, 0].set_xlabel('Epoch')
        axes[0, 0].set_ylabel('Accuracy')
        axes[0, 0].grid(True)
        axes[0, 0].legend(loc='lower right')
        axes[0, 0].set_xticks(self.epochs if len(self.epochs) < 15 else self.epochs[::2])

        # Plot Onsets Accuracy
        axes[0, 1].plot(self.epochs, self.epoch_data['onsets_output_accuracy'], 'b-o', label='Training Onsets Accuracy')
        axes[0, 1].plot(self.epochs, self.epoch_data['val_onsets_output_accuracy'], 'r-x', label='Validation Onsets Accuracy')
        axes[0, 1].set_title('Onsets Accuracy')
        axes[0, 1].set_xlabel('Epoch')
        axes[0, 1].set_ylabel('Accuracy')
        axes[0, 1].grid(True)
        axes[0, 1].legend(loc='lower right')
        axes[0, 1].set_xticks(self.epochs if len(self.epochs) < 15 else self.epochs[::2])
        
        # Plot Total Loss
        axes[1, 0].plot(self.epochs, self.epoch_data['loss'], 'b-o', label='Total Training Loss')
        axes[1, 0].plot(self.epochs, self.epoch_data['val_loss'], 'r-x', label='Total Validation Loss')
        axes[1, 0].set_title('Total Loss')
        axes[1, 0].set_xlabel('Epoch')
        axes[1, 0].set_ylabel('Loss')
        axes[1, 0].grid(True)
        axes[1, 0].legend(loc='upper right')
        axes[1, 0].set_xticks(self.epochs if len(self.epochs) < 15 else self.epochs[::2])

        # Plot Individual Losses (Note vs Onsets)
        axes[1, 1].plot(self.epochs, self.epoch_data['note_output_loss'], 'g-o', label='Training Note Loss')
        axes[1, 1].plot(self.epochs, self.epoch_data['val_note_output_loss'], 'g--x', label='Validation Note Loss')
        axes[1, 1].plot(self.epochs, self.epoch_data['onsets_output_loss'], 'm-o', label='Training Onsets Loss')
        axes[1, 1].plot(self.epochs, self.epoch_data['val_onsets_output_loss'], 'm--x', label='Validation Onsets Loss')
        axes[1, 1].set_title('Individual Task Losses')
        axes[1, 1].set_xlabel('Epoch')
        axes[1, 1].set_ylabel('Loss')
        axes[1, 1].grid(True)
        axes[1, 1].legend(loc='upper right')
        axes[1, 1].set_xticks(self.epochs if len(self.epochs) < 15 else self.epochs[::2])

        plt.tight_layout(rect=[0, 0.03, 1, 0.95])
        
        # Save the plot to a file inside the timestamped directory
        plot_filename = os.path.join(self.plot_save_dir, f"training_plot.png")
        plt.savefig(plot_filename, dpi=150)
        plt.close(fig) # Close the figure to free up memory

        plt.show()

    def on_train_end(self, logs=None):
        print("Training finished. Final plot above.")











# --- 2. Compile the Model (Updated for multiple outputs) ---
cnn_model = build_cnn_model(INPUT_SHAPE, OUTPUT_DIM_NOTES, OUTPUT_DIM_ONSETS)

# Define loss functions for each output
losses = {
    'note_output': 'binary_crossentropy',
    'onsets_output': 'binary_crossentropy'
}

# Define metrics for each output
metrics = {
    'note_output': 'accuracy', # or tf.keras.metrics.BinaryAccuracy()
    'onsets_output': 'accuracy' # or tf.keras.metrics.BinaryAccuracy()
}

# Optional: Assign loss weights if one task is more important or if classes are highly imbalanced
# For onsets, you might want to give it a higher weight because onset events are typically sparse.
loss_weights = {
    'note_output': 1.0,
    'onsets_output': 1.0 # Give onset prediction more importance, adjust as needed
}

cnn_model.compile(optimizer=optimizers.Adam(learning_rate=LEARNING_RATE,clipnorm=1.0),
                  loss=losses,
                  loss_weights=loss_weights, # Include loss weights
                  metrics=metrics)

cnn_model.summary()

# --- 3. Data Loading and Preparation (Stream from Disk - .npy files) ---

# Get lists of all input and output file paths
input_filepaths = sorted(glob.glob(os.path.join(input_data_dir, '*.npy')))
output_filepaths = sorted(glob.glob(os.path.join(output_data_dir, '*.npy'))) # For notes
onsets_filepaths = sorted(glob.glob(os.path.join(onsets_data_dir, '*.npy'))) # For onsets

total_samples_on_disk = len(input_filepaths)
if total_samples_on_disk == 0:
    print(f"ERROR: No .npy files found in {input_data_dir}. Please run the serialization script first.")
    exit()
if total_samples_on_disk != len(output_filepaths) or total_samples_on_disk != len(onsets_filepaths):
    print("ERROR: Mismatch in number of input, note output, or onsets output files.")
    exit()

print(f"Found {total_samples_on_disk} files on disk.")
class_weights_onsets = {
    0.0: 0.001,#0.008081424936386769,
    1.0: 1.0-0.001   # Higher weight for Class 1 (e.g., the minority class)
}

print("Class Weights for onsets_output:")
print(class_weights_onsets)

# Function to load a single image and its labels from file paths (Updated)
def load_input_from_files(input_path_tensor):
    # Convert TensorFlow string tensors to Python strings
    input_path = input_path_tensor.numpy().decode('utf-8')

    
    # Load the NumPy arrays
    image = np.load(input_path).astype(np.float32).reshape(INPUT_SHAPE)

    image = tf.ensure_shape(image, INPUT_SHAPE)

  
    return image


def load_notes_from_files( note_output_path_tensor):
    # Convert TensorFlow string tensors to Python strings

    note_output_path = note_output_path_tensor.numpy().decode('utf-8')

    
    # Load the NumPy arrays

    note_label = np.load(note_output_path).astype(np.float32).reshape(OUTPUT_DIM_NOTES)

    # Ensure shapes for TensorFlow

    note_label = tf.ensure_shape(note_label, (OUTPUT_DIM_NOTES,))

  
    return  note_label

def load_onsets_from_files(onset_path_tensor):
    # Convert TensorFlow string tensors to Python strings

    onset_path = onset_path_tensor.numpy().decode('utf-8')
    
    # Load the NumPy arrays

    onset_label = np.load(onset_path).astype(np.float32).reshape(OUTPUT_DIM_ONSETS)
    onset_weights=np.vectorize(class_weights_onsets.get)(onset_label)
    # onset_locations=np.where(onset_label==0.25)[0]
    # if(len(onset_locations)>0):
    #     loc=onset_locations[0]
    #     print('onset at index '+str(loc)+' weight: '+str(onset_weights[loc]))
    # Ensure shapes for TensorFlow

    onset_label = tf.ensure_shape(onset_label, (OUTPUT_DIM_ONSETS,))
    onset_weights = tf.convert_to_tensor(onset_weights, dtype=tf.float32)
    onset_weights=tf.ensure_shape(onset_weights, (OUTPUT_DIM_ONSETS,))
  
    return  onset_label,onset_weights



# TensorFlow wrapper function (Updated)
def tf_load_sample_from_files(ipath, nopath, opath):
    image = tf.py_function(
        load_input_from_files, [ipath], [tf.float32]
    )[0]
    note_label = tf.py_function(
        load_notes_from_files, [ nopath], [tf.float32]
    )[0]
    
    onset_label,onset_weights = tf.py_function(
        load_onsets_from_files, [opath], [tf.float32,tf.float32]
    )

    # Explicitly set shapes here!
    image.set_shape(INPUT_SHAPE)
    note_label.set_shape((OUTPUT_DIM_NOTES,))
    onset_label.set_shape((OUTPUT_DIM_ONSETS,))
    onset_weights.set_shape((OUTPUT_DIM_ONSETS,))
    #onsets_sample_weight.set_shape(()) # Scalar weight
    
    # Return as (input_data, (output1_labels, output2_labels))
    return (image, (note_label,onset_label),(None,onset_weights)
            )

# Create a dataset from the lists of file paths
dataset = tf.data.Dataset.from_tensor_slices((input_filepaths, output_filepaths, onsets_filepaths))

# Shuffle the list of paths first
dataset = dataset.shuffle(buffer_size=total_samples_on_disk) # Buffer size for shuffling paths

# Split the dataset into training and validation subsets based on indices
split_ratio = 0.7
num_train = int(total_samples_on_disk * split_ratio)

train_dataset = dataset.take(num_train)



val_dataset = dataset.skip(num_train)

# Map the loading function to the datasets using tf.py_function
train_dataset = train_dataset.map(
    tf_load_sample_from_files,
    num_parallel_calls=tf.data.AUTOTUNE # Load in parallel threads/processes
)
val_dataset = val_dataset.map(
    tf_load_sample_from_files,
    num_parallel_calls=tf.data.AUTOTUNE
)

# Apply batching and prefetching
train_dataset = train_dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)
val_dataset = val_dataset.batch(BATCH_SIZE).prefetch(tf.data.AUTOTUNE)

# --- 3. Configure Callbacks (Updated monitor names) ---
early_stopping_acc = EarlyStopping(
    monitor='val_note_output_accuracy', # Monitor validation accuracy for the 'note_output'
    patience=10,            # Number of epochs with no improvement
    mode='max',             # 'max' because we want to maximize accuracy
    verbose=1,              # Log when training stops
    restore_best_weights=True # Restore weights from the epoch with the best monitored value.
)

model_checkpoint_acc = ModelCheckpoint(
    'best_model_by_note_acc.keras', # A different filename
    monitor='val_note_output_accuracy',
    mode='max',
    save_best_only=True, # Only save when validation accuracy improves
    verbose=1
)
# --- NEW: ReduceLROnPlateau Callback ---
reduce_lr_on_plateau = ReduceLROnPlateau(
    monitor='val_note_output_accuracy', # Monitor the same metric as EarlyStopping
    factor=0.5,           # Reduce learning rate by half
    patience=5,           # Number of epochs with no improvement before reducing LR
    mode='max',           # 'max' because we want to maximize accuracy
    min_delta=0.0001,     # Minimum change to qualify as an improvement
    cooldown=0,           # Number of epochs to wait before resuming normal operation after LR has been reduced
    min_lr=0.00001,       # Lower bound on the learning rate
    verbose=1
)
jupyter_live_plot = JupyterLivePlottingCallback(fig_title="Keras Training Progress (Multi-Output)")

# --- 4. Training the Model ---
print("\n--- Training Pure CNN Model (Multi-Output) ---")
try:
    history_cnn = cnn_model.fit(train_dataset,
                                epochs=EPOCHS,
                                validation_data=val_dataset,
                                callbacks=[model_checkpoint_acc, early_stopping_acc, jupyter_live_plot,reduce_lr_on_plateau])
    # You might want to save the final weights too, or rely on ModelCheckpoint
    cnn_model.save_weights('guitarmidi-multi-output-final.weights.h5')
    cnn_model.save('guitarmidi.keras')
    print("Final model weights saved successfully!")
except Exception as e:
    print(f"An error occurred during training: {e}")

2025-07-08 00:00:10.743203: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1751925610.757563 3881654 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1751925610.762043 3881654 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1751925610.773344 3881654 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1751925610.773358 3881654 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1751925610.773359 3881654 computation_placer.cc:177] computation placer alr

Mixed precision policy set to 'mixed_float16'.
Memory growth enabled for GPUs.
TensorFlow version: 2.19.0


I0000 00:00:1751925612.549164 3881654 gpu_device.cc:2019] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 1693 MB memory:  -> device: 0, name: Quadro RTX 4000, pci bus id: 0000:08:00.0, compute capability: 7.5


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_features      │ (None, 256, 312,  │          0 │ -                 │
│ (InputLayer)        │ 1)                │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 256, 312,  │      1,600 │ input_features[0… │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 256, 312,  │        128 │ conv2d[0][0]      │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation          │ (None, 256, 312,  │          0 │ batch_normalizat… │
│ (Activation)        │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 128, 156,  │          0 │ activation[0][0]  │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ spatial_dropout2d   │ (None, 128, 156,  │          0 │ max_pooling2d[0]… │
│ (SpatialDropout2D)  │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 128, 156,  │    100,416 │ spatial_dropout2… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 128, 156,  │        256 │ conv2d_1[0][0]    │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_1        │ (None, 128, 156,  │          0 │ batch_normalizat… │
│ (Activation)        │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_1     │ (None, 64, 78,    │          0 │ activation_1[0][… │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ spatial_dropout2d_1 │ (None, 64, 78,    │          0 │ max_pooling2d_1[… │
│ (SpatialDropout2D)  │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 64, 78,    │    401,536 │ spatial_dropout2… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 64, 78,    │        512 │ conv2d_2[0][0]    │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_2        │ (None, 64, 78,    │          0 │ batch_normalizat… │
│ (Activation)        │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_2     │ (None, 32, 39,    │          0 │ activation_2[0][… │
│ (MaxPooling2D)      │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ spatial_dropout2d_2 │ (None, 32, 39,    │          0 │ max_pooling2d_2[… │
│ (SpatialDropout2D)  │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ global_average_poo… │ (None, 128)       │          0 │ spatial_dropout2

...

   2/2150 ━━━━━━━━━━━━━━━━━━━━ 2:43 76ms/step - loss: 0.8412 - note_output_accuracy: 0.0156 - note_output_loss: 0.8407 - onsets_output_accuracy: 0.8281 - onsets_output_loss: 5.4093e-04       

I0000 00:00:1751925662.130428 3881765 device_compiler.h:188] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


1882/2150 ━━━━━━━━━━━━━━━━━━━━ 16s 60ms/step - loss: 0.1414 - note_output_accuracy: 0.1877 - note_output_loss: 0.1384 - onsets_output_accuracy: 0.3907 - onsets_output_loss: 0.0030

KeyboardInterrupt: 

: 

In [ ]:
import matplotlib.pyplot as plt

epochs = range(1, len(history_cnn.history['loss']) + 1)

fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle("Training Metrics")

# Note Accuracy
axes[0, 0].plot(epochs, history_cnn.history['note_output_accuracy'], 'b-o', label='Training Note Accuracy')
axes[0, 0].plot(epochs, history_cnn.history['val_note_output_accuracy'], 'r-x', label='Validation Note Accuracy')
axes[0, 0].set_title('Note Accuracy')
axes[0, 0].set_xlabel('Epoch')
axes[0, 0].set_ylabel('Accuracy')
axes[0, 0].grid(True)
axes[0, 0].legend(loc='lower right')

# Onsets Accuracy
axes[0, 1].plot(epochs, history_cnn.history['onsets_output_accuracy'], 'b-o', label='Training Onsets Accuracy')
axes[0, 1].plot(epochs, history_cnn.history['val_onsets_output_accuracy'], 'r-x', label='Validation Onsets Accuracy')
axes[0, 1].set_title('Onsets Accuracy')
axes[0, 1].set_xlabel('Epoch')
axes[0, 1].set_ylabel('Accuracy')
axes[0, 1].grid(True)
axes[0, 1].legend(loc='lower right')

# Total Loss
axes[1, 0].plot(epochs, history_cnn.history['loss'], 'b-o', label='Total Training Loss')
axes[1, 0].plot(epochs, history_cnn.history['val_loss'], 'r-x', label='Total Validation Loss')
axes[1, 0].set_title('Total Loss')
axes[1, 0].set_xlabel('Epoch')
axes[1, 0].set_ylabel('Loss')
axes[1, 0].grid(True)
axes[1, 0].legend(loc='upper right')

# Individual Losses
axes[1, 1].plot(epochs, history_cnn.history['note_output_loss'], 'g-o', label='Training Note Loss')
axes[1, 1].plot(epochs, history_cnn.history['val_note_output_loss'], 'g--x', label='Validation Note Loss')
axes[1, 1].plot(epochs, history_cnn.history['onsets_output_loss'], 'm-o', label='Training Onsets Loss')
axes[1, 1].plot(epochs, history_cnn.history['val_onsets_output_loss'], 'm--x', label='Validation Onsets Loss')
axes[1, 1].set_title('Individual Task Losses')
axes[1, 1].set_xlabel('Epoch')
axes[1, 1].set_ylabel('Loss')
axes[1, 1].grid(True)
axes[1, 1].legend(loc='upper right')

plt.tight_layout(rect=[0, 0.03, 1, 0.95])
plt.show()


In [4]:
cnn_model.save('guitarmidi.keras')